In [6]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import aiohttp
import asyncio

In [12]:
async def get_kerings_data():
    """Retrieve all bagger project records from the ArcGIS REST service.

    The function first requests the total number of available records and
    subsequently retrieves all features in batches of 1,000 records. Results
    are fetched asynchronously using ``aiohttp`` and combined into a single
    pandas DataFrame.

    Returns:
        pd.DataFrame: Flattened DataFrame containing all feature attributes
            returned by the ArcGIS service.

    Notes:
        - Requests are limited to a maximum of 15 concurrent connections.
        - Feature attributes are normalized using ``pd.json_normalize``.
        - The ArcGIS service is queried using result offsets in increments
          of 1,000 records until all available records are retrieved.

    Raises:
        requests.RequestException: If the initial count request fails.
        aiohttp.ClientError: If one of the asynchronous requests fails.
        KeyError: If the count response does not contain a ``count`` field.
    """
    
    url = f"https://rijnland.enl-mcs.nl/arcgis/rest/services/Regionale_kering/MapServer/0/query?where=1=1&f=geojson&returnCountOnly=true"
    references = []
    response_count = requests.get(url)
    max_count = response_count.json()['count']
    urls = [f"https://rijnland.enl-mcs.nl/arcgis/rest/services/Regionale_kering/MapServer/0/query?where=1=1&f=json&outFields=*&resultOffset={i}" for i in range(0, max_count, 1000)]

    sem = asyncio.Semaphore(15)  # max concurrent requests

    async def fetch_one(session, url):
        async with sem:
            async with session.get(url) as response:
                return await response.json()

    references = []
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(
            *(fetch_one(session, url) for url in urls)
        )

        for result in results:
            references.extend(result.get("features", []))
    return pd.json_normalize(references)

In [13]:
test = await get_kerings_data()

In [14]:
test

,attributes.ORACLE_OBJECTID,attributes.CODE,attributes.NAAM,attributes.OPMERKING,attributes.GEOKPOBJECTID,attributes.LAST_EDITED_DATE,attributes.XCOORDINAAT,attributes.YCOORDINAAT,attributes.HYPERLINK,attributes.STATUSOBJECT,...,attributes.INWINNENDEINSTANTIE,attributes.METENDEINSTANTIE,attributes.INWINNINGSMETHODE,attributes.DIMENSIE,attributes.NAUWKEURIGHEIDXY,attributes.NAUWKEURIGHEIDZ,attributes.DATUMINWINNING,attributes.SHAPE_LEN,attributes.OBJECTID2,geometry.paths
0,16,079-042-00014,Lageveensepolder,NaN,8,1784015809000,96333.15,475766.26,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,1,"[[[96171.91099999845, 475517.9539999999], [961..."
1,17,093-042-00069,Polder Nieuwkoop,NaN,256,1784133089000,113469.15,466374.03,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,2,"[[[113395.10599999875, 466516.29300000146], [1..."
2,18,079-042-00015,Lageveensepolder,NaN,9,1784015809000,96519.62,476053.30,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,3,"[[[96494.39699999988, 476014.5639999993], [964..."
3,19,176-042-00015,Zwetterpolder,NaN,98,1784015809000,93129.37,473136.86,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,4,"[[[93087.875, 473109.18200000003], [93088.8330..."
4,20,074-042-00014,Kooipolder,NaN,976,1784015809000,96739.89,469604.55,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,5,"[[[96889.46599999815, 469751.12000000104], [96..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3597,9233,000-042-00001,Noordelijke toegang Waaiersluis,informatie over deze waterkering is beschikbaa...,13762,1784015809000,109992.39,446979.97,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,3598,"[[[109987.45800000057, 446947.83900000155], [1..."
3598,2223,AGV-A531_002,Lascarpad,Informatie over deze waterkering is opgenomen ...,8642,1787328995000,112060.23,486298.02,None,gerealiseerd,...,"Waterschap Amstel, Gooi en Vecht",None,NaN,2D,NaN,NaN,1.787270e+12,0,3599,"[[[111992.08689999953, 485855.9721000008], [11..."
3599,3572,249-042-00193,Polder de Noordplas,NaN,1825,1784131550000,96218.78,455152.50,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,3600,"[[[96730.8900000006, 454800.51399999857], [967..."
3600,3573,040-042-00049,Gogerpolder,NaN,1346,1784131550000,103237.26,470011.98,None,gerealiseerd,...,NaN,None,NaN,NaN,NaN,NaN,NaN,0,3601,"[[[103033.2490000017, 469980.14400000125], [10..."
